# Unit 4：MNE 中 ICA 实战

## 目标
- 掌握 MNE ICA 的标准流水线
- 学会 ICA 拟合、成分可视化、识别与剔除
- 理解每个步骤的注意事项


### 4.1 ICA 处理流水线

```
原始 EEG
  ↓
① 滤波 (1-40 Hz 带通)      ← 去漂移和高频噪声，利于 ICA 收敛
  ↓
② 剔除大伪迹段 (reject)    ← 大幅值异常会绑架 ICA 成分
  ↓
③ ICA 拟合                 ← 数据干净才能得到好分解
  ↓
④ 成分可视化 & 识别         ← 地形图 + 时间序列 + 功率谱
  ↓
⑤ 剔除伪迹成分             ← apply 到原始/滤波后数据
  ↓
干净 EEG
```


### 4.2 步骤①：滤波

**为什么 ICA 前要滤波？**
- 高通（1 Hz）：去除 DC 漂移，漂移不是独立成分
- 低通（40 Hz）：去除肌电高频噪声，它们也不满足 ICA 的线性混合假设


In [ ]:
import mne
import numpy as np
import matplotlib.pyplot as plt

sample_data_dir = mne.datasets.sample.data_path()
raw_fname = sample_data_dir / 'MEG' / 'sample' / 'sample_audvis_raw.fif'

raw = mne.io.read_raw_fif(raw_fname, preload=True)
raw.pick(['eeg'])
montage = mne.channels.make_standard_montage('standard_1020')
raw.set_montage(montage)

# 带通滤波
raw_filtered = raw.copy().filter(l_freq=1.0, h_freq=40.0)

# 对比
fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
before = raw.get_data(picks='EEG 001')[0, :2000]
after  = raw_filtered.get_data(picks='EEG 001')[0, :2000]
times  = raw.times[:2000]
axes[0].plot(times, before*1e6, linewidth=0.5)
axes[0].set_ylabel('μV'); axes[0].set_title('滤波前（注意漂移和高频毛刺）')
axes[1].plot(times, after*1e6, linewidth=0.5, color='green')
axes[1].set_ylabel('μV'); axes[1].set_title('滤波后 (1-40 Hz)')
axes[1].set_xlabel('时间 (s)')
plt.tight_layout(); plt.show()


### 4.3 步骤②：剔除大伪迹段

FastICA 对大幅值异常敏感，大伪迹会把整个成分"绑架"成那个瞬间的形状。

最佳实践：用 `reject` 参数排除峰峰值过大的时间窗口。


In [ ]:
# 基于峰峰值剔除：任何通道在 1 秒内峰峰值超过 200 μV 的段被排除
reject_criteria = dict(eeg=200e-6)  # 200 μV

# 创建 epochs 只是为了用 reject（不用实际上做 epoch）
events = mne.make_fixed_length_events(raw_filtered, duration=1.0)
epochs = mne.Epochs(raw_filtered, events, tmin=0, tmax=0.99,
                     baseline=None, reject=reject_criteria, preload=False)
print(f"原始 1s 段数: {len(events)}")
print(f"剔除坏段后: {len(epochs)} ({len(events)-len(epochs)} 段被排除)")
print(f"保留数据占比: {len(epochs)/len(events)*100:.1f}%")


### 4.4 步骤③：ICA 拟合

**成分数量怎么选？** 这里有 60 个 EEG 通道：
- 太少（如 10 个）→ 伪迹和脑信号挤在同一成分，剔除时误伤脑信号
- 太多（如 40 个）→ 脑信号过度拆分，人工识别负担大
- **推荐 15-25 个**，本教程用 15 个


In [ ]:
from mne.preprocessing import ICA

n_components = 15
ica = ICA(
    n_components=n_components,
    method='fastica',
    random_state=42,
    max_iter='auto',
    fit_params=dict(extended=True)  # 扩展 FastICA，更稳定
)

print(f"开始 ICA 拟合（{n_components} 个成分，{len(raw_filtered.ch_names)} 个通道）...")
ica.fit(raw_filtered)
print(f"✅ 完成！迭代次数: {ica.n_iter_}")
print(f"解释方差比: {np.sum(ica.pca_explained_variance_[:n_components]) / np.sum(ica.pca_explained_variance_):.1%}")


### 4.5 步骤④：成分可视化与识别

三种互补视角来识别伪迹成分：
1. **地形图**（空间分布）—— 前额集中 → 眨眼
2. **时间序列**（时域激活）—— 间歇脉冲 → 眨眼
3. **功率谱**（频域特征）—— 低频主导 → 眼动


In [ ]:
# 方式一：地形图 —— 看空间分布
ica.plot_components(picks=range(n_components))
print("观察要点：前额高亮 → 眨眼/眼动；颞部集中 → 肌电；全头分散 → 可能是心跳")


In [ ]:
# 方式二：时间序列 —— 看激活模式
ica.plot_sources(raw_filtered, picks=range(min(10, n_components)))
print("观察要点：间歇性大尖峰 → 眨眼；规律脉冲 → 心跳；持续高频 → 肌电")


In [ ]:
# 方式三：综合属性（地形图 + 时间序列 + 功率谱三合一）
# 先看成分 0，你自己看其他可疑成分
ica.plot_properties(raw_filtered, picks=[0])


### 4.6 识别伪迹成分的对照表

| 特征 | 眨眼 | 眼动 | 心跳 | 肌电 |
|------|------|------|------|------|
| 地形图 | 前额集中 🔴 | 单侧/双侧前部 | 分散或特定 | 颞部局部 |
| 时间序列 | 间歇大脉冲 | 缓慢方波/漂移 | 规律尖峰 | 持续高频噪声 |
| 功率谱 | <5 Hz 主导 | <4 Hz 主导 | 5-20 Hz 有峰 | >20 Hz 隆起 |


In [ ]:
# 自动辅助：用 Fp1 找眼动相关成分
eog_indices, eog_scores = ica.find_bads_eog(
    raw, ch_name='EEG 001', threshold=3.0
)
print(f"自动检测到的眼动成分: {eog_indices}")
print(f"与 Fp1 的相关性得分: {np.array2string(eog_scores, precision=3)}")

# ⚠️ 不要盲目相信自动检测！要结合地形图和时间序列人工确认。
print("\n💡 记住这些成分编号，和你的肉眼观察对比。Unit 5 会实际剔除。")


### 4.7 步骤⑤：剔除伪迹成分

```python
# 标记
ica.exclude = [0, 2]   # 根据前面的观察确定

# 应用到数据
raw_clean = ica.apply(raw_filtered.copy())

# 对比
raw_filtered.plot(n_channels=10, title='ICA 前')
raw_clean.plot(n_channels=10, title='ICA 后')
```

### ⚠️ 关键注意事项

1. **叠加因素**：先滤波再 ICA，但 `apply` 可以用在原始数据上
2. **适度剔除**：通常剔除 5-20% 的成分
3. **反复验证**：剔除后检查时间序列和地形图，确认伪迹减少
4. **随机种子**：设 `random_state` 保证可复现

### 🤔 思考题

- 为什么滤波要在 ICA 之前？（想想中心极限定理和非高斯性）
- 成分 0 是眨眼，成分 5 也是眨眼 — 为什么会分成两个成分？

→ 进入 **Unit 5：完整演练**
